# RAG Pipeline: PDF Question Answering System

This notebook builds a complete Retrieval-Augmented Generation (RAG) pipeline:

**PDF documents → chunks → embeddings → vector store (ChromaDB) → semantic retrieval → LLM-generated answer**

Stack used: LangChain, SentenceTransformers, ChromaDB, Groq (LLM inference).

## Setup

In [5]:
import os
os.makedirs("data/pdfs", exist_ok=True)
print("folder ready")

folder ready


In [6]:
os.listdir("data/pdfs")

['research.pdf']

In [4]:
!pip install langchain langchain-core langchain-community langchain-groq pypdf pymupdf sentence-transformers chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61

In [7]:
from langchain_core.documents import Document

# Document is the standard container LangChain uses to hold text + metadata
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

## Ingestion Pipeline

Flow: **Data → Documents → Chunks → Embeddings → Vector Store**

### Step 1: Load documents

In [8]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs(folder_path="data/pdfs"):
    """Load every PDF in a folder into a single list of Document objects (one per page)."""
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

/tmp/ipykernel_1288/764971693.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [9]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 21


### Step 2: Split into chunks

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    """Break large documents into overlapping chunks so retrieval stays precise."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [11]:
chunks = split_docs(all_pdf_documents)
print("total chunks:", len(chunks))

total chunks: 244


### Step 3: Generate embeddings

In [12]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Converts text into numeric vectors so semantic similarity can be computed."""

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [13]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_1288/740053779.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Step 4: Store in a vector database (ChromaDB)

In [14]:
import chromadb
import uuid

class VectorStoreManager:
    """Wraps a persistent ChromaDB collection for storing and querying embeddings."""

    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # add() is called once, after the loop, with the full batch
        # (previously this was indented inside the loop, causing duplicate inserts)
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [15]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [16]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 244


## Retrieval Pipeline

Given a user query, find the most semantically similar chunks from the vector store.

In [17]:
class RAGRetriever:
    """Retrieves the top-k most relevant chunks for a given query."""

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search in Chroma
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents found")

        return retrieved_docs

In [18]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [19]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents


[]

## Generation Pipeline

This is the "G" in RAG: take the retrieved chunks, put them into a prompt as context,
and ask an LLM to generate a grounded answer. We use Groq here because it has a fast,
free-tier API that's easy to set up for a student project — swap in OpenAI/Gemini/local
model if you prefer.

Get a free API key from https://console.groq.com/keys

In [20]:
from getpass import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [21]:
from langchain_groq import ChatGroq

class RAGPipeline:
    """Combines retrieval with LLM generation to answer questions grounded in the PDFs."""

    def __init__(self, retriever, model_name="llama-3.1-8b-instant", top_k=5):
        self.retriever = retriever
        self.top_k = top_k
        self.llm = ChatGroq(model=model_name, temperature=0)

    def _build_prompt(self, query, retrieved_docs):
        context = "\n\n".join(
            f"[Source {i+1}] {doc['document']}" for i, doc in enumerate(retrieved_docs)
        )

        prompt = f"""You are a helpful assistant answering questions using only the provided context.
If the answer is not contained in the context, say you don't know instead of guessing.

Context:
{context}

Question: {query}

Answer:"""
        return prompt

    def answer(self, query):
        retrieved_docs = self.retriever.retrieve(query, top_k=self.top_k)

        if not retrieved_docs:
            return {"answer": "No relevant context found in the documents.", "sources": []}

        prompt = self._build_prompt(query, retrieved_docs)
        response = self.llm.invoke(prompt)

        sources = [doc["metadata"].get("source", "unknown") for doc in retrieved_docs]

        return {
            "answer": response.content,
            "sources": sources,
            "retrieved_chunks": retrieved_docs
        }

In [22]:
rag_pipeline = RAGPipeline(rag_retriever)

result = rag_pipeline.answer("What is encoder decoder")

print("ANSWER:\n", result["answer"])
print("\nSOURCES:", result["sources"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents
ANSWER:
 No relevant context found in the documents.

SOURCES: []
